# Court LLM Descriptor Extractor — Kaggle GPU Only

This notebook is intentionally **not** the final enrichment pipeline.

It runs only the expensive LLM step on Kaggle GPU and writes **raw minimal semantic descriptors**:

```text
court_considerations.csv
→ Qwen3-8B-AWQ
→ court_llm_descriptors_*.jsonl
```

Do **not** do anchor normalization here. Do **not** build final retrieval views here.

After downloading the output JSONL, run local CPU scripts:

```text
scripts/court_enrichment_profile.py
scripts/court_enrichment_normalizer.py
scripts/finalize_court_enrichment_from_llm.py
```

The local finalizer will merge the raw LLM descriptors with `court_considerations.csv`, normalize anchors, correct role/outcome, build retrieval views, and produce final production JSONL.


## Dual T4 mode

This version defaults to `cfg.gpu_mode = 'dual_worker'`, which launches two independent vLLM workers: GPU 0 handles the first half of selected rows and GPU 1 handles the second half. This is usually faster than tensor parallelism for Qwen3-8B-AWQ because the model fits on one T4.


In [1]:
!pip install -q -U "transformers>=4.45.0" accelerate safetensors pandas tqdm gptqmodel
!pip install -q -U "vllm>=0.6.0" || true
!pip uninstall -y flashinfer flashinfer-python || true

print("Setup done. Now restart the Kaggle session/kernel, then run from the next cell.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.20.0 requires flashinfer-python==0.6.8.post1, which is not installed.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
opentelemetry-proto 1.41.1 requires protobuf<7.0,>=5.0, but you have protobuf 7.34.1 which is incompatible.
google-cloud-videointelligence 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-cloud-vision 3.12.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
grpc-google-iam-v1 0.14.3 requires protobuf!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but y

In [2]:
# Cell 1 — Imports

from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Optional
from collections import Counter
import os
import re
import gc
import ast
import json
import time
import traceback

import pandas as pd
from tqdm.auto import tqdm

try:
    import torch
except Exception:
    torch = None

print('Imports OK')
if torch is not None and torch.cuda.is_available():
    print('CUDA devices:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        free, total = torch.cuda.mem_get_info(i)
        print(f'GPU {i}: {props.name}; free={free/1024**3:.2f} GiB / total={total/1024**3:.2f} GiB')


Imports OK
CUDA devices: 2
GPU 0: Tesla T4; free=14.46 GiB / total=14.56 GiB
GPU 1: Tesla T4; free=14.46 GiB / total=14.56 GiB


In [3]:
# Cell 2 — Config

@dataclass
class Config:
    # Input CSV
    input_csv: str = '/kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv'
    fallback_input_csv: str = 'court_considerations.csv'

    # Local Kaggle model path
    model_name: str = '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1'

    # Row selection
    start: int = 0
    limit: int = 50
    sample_random: bool = False
    random_seed: Optional[int] = 42
    min_text_chars: int = 80
    max_text_chars: int = 3200

    # GPU/vLLM
    gpu_mode: str = 'dual_worker'   # 'single', 'tp2', or 'dual_worker' (fastest for 2xT4 when model fits on 1 GPU)
    tensor_parallel_size: int = 1
    gpu_memory_utilization: float = 0.88
    max_model_len: int = 4096
    max_num_seqs: int = 16
    batch_size: int = 8
    enforce_eager: bool = False
    quantization: str = 'awq_marlin'
    disable_custom_all_reduce: bool = True
    force_triton_attention: bool = True

    # Structured output is disabled because Kaggle vLLM v0.20 can crash with:
    # AttributeError("'dict' object has no attribute '_backend'")
    use_structured_outputs: bool = False

    # Generation
    max_new_tokens: int = 384
    retry_max_new_tokens: int = 512
    max_retries: int = 1
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.02
    enable_thinking: bool = False

    # Output
    output_dir: str = '/kaggle/working'
    include_raw_output_on_success: bool = False

cfg = Config()

# Apply GPU mode before loading vLLM.
# - single: one vLLM engine on GPU 0
# - tp2: one vLLM engine split across both GPUs; useful only when the model/context does not fit on one GPU
# - dual_worker: fastest for this 8B AWQ workload; two independent vLLM engines, one per T4, each processing half the rows
if cfg.gpu_mode == 'single':
    os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
    cfg.tensor_parallel_size = 1
elif cfg.gpu_mode == 'tp2':
    os.environ.pop('CUDA_VISIBLE_DEVICES', None)
    cfg.tensor_parallel_size = 2
    cfg.disable_custom_all_reduce = True
elif cfg.gpu_mode == 'dual_worker':
    # Keep both GPUs visible in the notebook controller.
    # The launcher cell will start two subprocesses with CUDA_VISIBLE_DEVICES=0 and CUDA_VISIBLE_DEVICES=1.
    os.environ.pop('CUDA_VISIBLE_DEVICES', None)
    cfg.tensor_parallel_size = 1
else:
    raise ValueError("cfg.gpu_mode must be 'single', 'tp2', or 'dual_worker'")

if cfg.force_triton_attention:
    os.environ.setdefault('VLLM_ATTENTION_BACKEND', 'TRITON_ATTN')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

model_path = Path(cfg.model_name)
if not model_path.exists():
    raise FileNotFoundError(f'Model path does not exist: {model_path}. Attach the Kaggle model dataset.')

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

end_idx = cfg.start + cfg.limit - 1 if cfg.limit else -1
suffix = f'{cfg.start:07d}_{end_idx:07d}' if cfg.limit else f'{cfg.start:07d}_all'
output_jsonl = out_dir / f'court_llm_descriptors_{suffix}.jsonl'
output_preview_csv = out_dir / f'court_llm_descriptors_{suffix}_preview.csv'
output_failures_jsonl = out_dir / f'court_llm_descriptors_{suffix}_failures.jsonl'
output_metrics_json = out_dir / f'court_llm_descriptors_{suffix}_metrics.json'

print(asdict(cfg))
print('Output JSONL:', output_jsonl)


{'input_csv': '/kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv', 'fallback_input_csv': 'court_considerations.csv', 'model_name': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1', 'start': 0, 'limit': 50, 'sample_random': False, 'random_seed': 42, 'min_text_chars': 80, 'max_text_chars': 3200, 'gpu_mode': 'single', 'tensor_parallel_size': 1, 'gpu_memory_utilization': 0.78, 'max_model_len': 4096, 'max_num_seqs': 8, 'batch_size': 4, 'enforce_eager': True, 'quantization': 'awq_marlin', 'disable_custom_all_reduce': True, 'force_triton_attention': True, 'use_structured_outputs': False, 'max_new_tokens': 384, 'retry_max_new_tokens': 512, 'max_retries': 1, 'temperature': 0.0, 'top_p': 1.0, 'repetition_penalty': 1.02, 'enable_thinking': False, 'output_dir': '/kaggle/working', 'include_raw_output_on_success': False}
Output JSONL: /kaggle/working/court_llm_descriptors_0000000_0000049.jsonl


In [4]:
# Cell 3 — Load CSV and select rows

def resolve_input_path() -> Path:
    candidates = [
        Path(cfg.input_csv),
        Path(cfg.fallback_input_csv),
        Path('/kaggle/working') / cfg.fallback_input_csv,
    ]
    for p in candidates:
        if p.exists():
            return p
    if Path('/kaggle/input').exists():
        for pat in ['**/court_considerations.csv', '**/court_consideration.csv']:
            found = sorted(Path('/kaggle/input').glob(pat))
            if found:
                return found[0]
    raise FileNotFoundError('Could not find court_considerations.csv')

input_path = resolve_input_path()
print('Using input:', input_path)

df = pd.read_csv(input_path)
print('Shape:', df.shape)
print('Columns:', list(df.columns))

citation_col = 'citation' if 'citation' in df.columns else df.columns[0]
text_col = 'text' if 'text' in df.columns else df.columns[1]

valid = df[df[citation_col].notna() & df[text_col].notna()].copy()
valid[text_col] = valid[text_col].astype(str)
valid['_text_len'] = valid[text_col].str.strip().str.len()
valid = valid[valid['_text_len'] >= cfg.min_text_chars].copy()

if cfg.sample_random:
    pool = valid.iloc[cfg.start:] if cfg.start else valid
    work_df = pool.sample(n=min(cfg.limit, len(pool)), random_state=cfg.random_seed)
else:
    end = None if not cfg.limit else cfg.start + cfg.limit
    work_df = valid.iloc[cfg.start:end]

work_df = work_df.reset_index(drop=False).rename(columns={'index': '_source_row'})
print('Selected rows:', len(work_df))
display(work_df[['_source_row', citation_col, text_col, '_text_len']].head(20))


Using input: /kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv
Shape: (2476315, 2)
Columns: ['citation', 'text']
Selected rows: 50


,_source_row,citation,text,_text_len
0,1,BGE 139 I 2 E. 2,Eventualiter sei die Rückweisung an die Vorins...,885
1,2,BGE 139 I 2 E. 5.1,"In der Sache ist vorweg zu prüfen, ob der Ents...",437
2,3,BGE 139 I 2 E. 5.2,Art. 34 Abs. 1 BV gewährleistet in allgemeiner...,242
3,4,BGE 139 I 2 E. 5.3,Im vorliegenden Fall geht es nicht um die Gült...,286
4,5,BGE 139 I 2 E. 7.1,S. 144) bestätigte das Verwaltungsgericht den ...,1493
5,6,BGE 139 I 2 E. 5.4,Strittig ist hier hingegen die Umsetzung der P...,183
6,7,BGE 139 I 2 E. 7.1,S. 70) dargestellten und im angefochtenen Ents...,690
7,8,BGE 139 I 2 E. 5.5,"Zu beachten ist sodann, dass nach der schwyzer...",904
8,9,BGE 139 I 2 E. 5.6,Die Umsetzung einer Planungsinitiative ist ver...,2405
9,10,BGE 139 I 2 E. 5.7,Die an der Volksabstimmung vom 26. November 20...,278


In [5]:
# Cell 4 — Minimal LLM descriptor schema and prompt

# The LLM intentionally does NOT produce anchors, outcomes, summaries, questions, or retrieval views.
# Local CPU scripts will build those deterministically later.

DESCRIPTOR_KEYS = [
    'legal_area',
    'primary_domain',
    'secondary_domain',
    'legal_domain_path',
    'topic',
    'subtopic',
    'micro_topic',
    'concepts_en',
    'terms_original',
    'doctrinal_rule',
    'legal_test',
    'fact_pattern_tags',
    'procedural_context',
    'paragraph_role',
    'authority_role',
    'specificity_score',
]

ROLE_VALUES = {
    'holding', 'reasoning', 'facts', 'procedural_history', 'legal_standard',
    'application', 'citation', 'costs', 'notification', 'disposition', 'neutral'
}

LLM_SCHEMA_HINT = {
    'legal_area': 'broad English legal area, max 6 words',
    'primary_domain': 'stable domain, max 6 words',
    'secondary_domain': 'narrow domain, max 8 words',
    'legal_domain_path': ['2-5 short taxonomy labels, broad to narrow'],
    'topic': 'specific topic, max 8 words',
    'subtopic': 'more specific subtopic, max 10 words',
    'micro_topic': 'most specific legal issue, max 14 words',
    'concepts_en': ['3-6 precise English legal concepts'],
    'terms_original': ['3-8 exact important source-language legal terms'],
    'doctrinal_rule': 'only if paragraph states a rule; otherwise empty; max 25 words',
    'legal_test': 'only if paragraph states/applies a test; otherwise empty; max 20 words',
    'fact_pattern_tags': ['0-5 concrete factual/procedural tags'],
    'procedural_context': 'short procedural posture if clear',
    'paragraph_role': 'holding|reasoning|facts|procedural_history|legal_standard|application|citation|costs|notification|disposition|neutral',
    'authority_role': ['0-3 legal value labels, e.g. legal_test, application_of_rule, background'],
    'specificity_score': 'number 0 to 1',
}

SYSTEM_PROMPT = '''You are a Swiss legal descriptor extractor.

Return exactly one compact JSON object.
Do not generate user questions.
Do not generate summaries.
Do not extract statute anchors.
Do not extract case anchors.
Do not infer final court outcome.
Do not build retrieval views.

Only extract query-neutral semantic descriptors that cannot be reliably obtained by static regex parsing.
Use English for classification fields.
Use exact German/French/Italian terms for terms_original.
If the paragraph is factual/procedural/boilerplate, keep doctrinal_rule and legal_test empty.
Use only information grounded in the citation text.
JSON only.'''

USER_TEMPLATE = '''Citation: {citation}

Text:
{text}

Required JSON shape:
{schema}

Return JSON only.'''

def trim_text(text: str, max_chars: int) -> str:
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if len(text) <= max_chars:
        return text
    head = max_chars // 2
    tail = max_chars - head
    return text[:head].rstrip() + ' ... [TRUNCATED] ... ' + text[-tail:].lstrip()

def build_user_prompt(citation: str, text: str) -> str:
    return USER_TEMPLATE.format(
        citation=str(citation),
        text=trim_text(text, cfg.max_text_chars),
        schema=json.dumps(LLM_SCHEMA_HINT, ensure_ascii=False),
    )


In [6]:
# Cell 5 — JSON parsing and descriptor normalization

def extract_json_object(raw: str) -> dict[str, Any]:
    if raw is None:
        raise ValueError('empty model output')
    s = str(raw).strip()
    s = re.sub(r'^\s*```(?:json)?\s*', '', s, flags=re.I)
    s = re.sub(r'\s*```\s*$', '', s)
    s = re.sub(r'<think>.*?</think>', '', s, flags=re.I | re.S).strip()

    start = s.find('{')
    if start < 0:
        raise ValueError(f'no JSON object start found: {s[:300]}')

    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_str:
            if esc:
                esc = False
            elif ch == '\\':
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    candidate = s[start:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        candidate = re.sub(r',\s*([}\]])', r'\1', candidate)
                        try:
                            return json.loads(candidate)
                        except Exception:
                            return ast.literal_eval(candidate)
    raise ValueError(f'no balanced JSON object found: {s[:700]}')


def clean_str(x: Any, max_chars: int = 240) -> str:
    s = re.sub(r'\s+', ' ', str(x or '')).strip()
    return s[:max_chars].rstrip()


def clean_list(x: Any, max_items: int, max_chars: int = 80) -> list[str]:
    if x is None:
        return []
    if isinstance(x, str):
        x = [x]
    if not isinstance(x, (list, tuple, set)):
        return []
    out, seen = [], set()
    for item in x:
        s = clean_str(item, max_chars=max_chars)
        if not s:
            continue
        key = s.casefold()
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
        if len(out) >= max_items:
            break
    return out


def normalize_descriptor(obj: dict[str, Any]) -> dict[str, Any]:
    d = {}
    d['legal_area'] = clean_str(obj.get('legal_area'), 80)
    d['primary_domain'] = clean_str(obj.get('primary_domain'), 80)
    d['secondary_domain'] = clean_str(obj.get('secondary_domain'), 100)
    d['legal_domain_path'] = clean_list(obj.get('legal_domain_path'), 5, 60)
    d['topic'] = clean_str(obj.get('topic'), 100)
    d['subtopic'] = clean_str(obj.get('subtopic'), 120)
    d['micro_topic'] = clean_str(obj.get('micro_topic'), 160)
    d['concepts_en'] = clean_list(obj.get('concepts_en'), 6, 70)
    d['terms_original'] = clean_list(obj.get('terms_original'), 8, 100)
    d['doctrinal_rule'] = clean_str(obj.get('doctrinal_rule'), 260)
    d['legal_test'] = clean_str(obj.get('legal_test'), 220)
    d['fact_pattern_tags'] = clean_list(obj.get('fact_pattern_tags'), 5, 70)
    d['procedural_context'] = clean_str(obj.get('procedural_context'), 120)

    role = clean_str(obj.get('paragraph_role'), 60).lower().replace(' ', '_').replace('-', '_')
    d['paragraph_role'] = role if role in ROLE_VALUES else 'neutral'
    d['authority_role'] = clean_list(obj.get('authority_role'), 3, 60)
    try:
        d['specificity_score'] = max(0.0, min(1.0, float(obj.get('specificity_score', 0))))
    except Exception:
        d['specificity_score'] = 0.0

    # Hard guarantee: forbidden/final fields never survive in raw descriptor.
    forbidden = {
        'statute_anchors', 'case_anchors', 'normalized_anchors', 'retrieval_views',
        'outcome_signal', 'query_phrases_en', 'natural_language_queries', 'legal_question',
        'summary_en', 'english_summary', 'enrichment_quality', 'anchor_quality_flags',
    }
    for k in forbidden:
        d.pop(k, None)
    return d


def empty_descriptor(error: str = '') -> dict[str, Any]:
    return {
        'legal_area': '',
        'primary_domain': '',
        'secondary_domain': '',
        'legal_domain_path': [],
        'topic': '',
        'subtopic': '',
        'micro_topic': '',
        'concepts_en': [],
        'terms_original': [],
        'doctrinal_rule': '',
        'legal_test': '',
        'fact_pattern_tags': [],
        'procedural_context': '',
        'paragraph_role': 'neutral',
        'authority_role': [],
        'specificity_score': 0.0,
        '_descriptor_error': error[:500],
    }


In [ ]:
# Cell 6b — Fast dual-T4 launcher (two independent vLLM workers)

# This is the fastest mode for Qwen3-8B-AWQ on 2xT4 because the model fits on one GPU.
# It starts two separate Python/vLLM processes:
#   GPU 0 -> first half of selected rows
#   GPU 1 -> second half of selected rows
# Each worker uses tensor_parallel_size=1, then this cell merges the two JSONL outputs.

if cfg.gpu_mode == 'dual_worker':
    import subprocess
    import sys
    import math
    import shlex
    from IPython.display import display

    if cfg.sample_random:
        raise ValueError("dual_worker currently expects cfg.sample_random=False so row ranges can be split deterministically.")

    total_rows = len(work_df)
    if total_rows == 0:
        raise ValueError("No rows selected; adjust cfg.start/cfg.limit/min_text_chars.")

    worker_script_path = Path(cfg.output_dir) / 'vllm_dual_worker.py'
    worker_script_path.write_text('# Cell 1 — Imports\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom dataclasses import dataclass, asdict\nfrom typing import Any, Optional\nfrom collections import Counter\nimport os\nimport re\nimport gc\nimport ast\nimport json\nimport time\nimport traceback\n\nimport pandas as pd\nfrom tqdm.auto import tqdm\n\ntry:\n    import torch\nexcept Exception:\n    torch = None\n\nprint(\'Imports OK\')\nif torch is not None and torch.cuda.is_available():\n    print(\'CUDA devices:\', torch.cuda.device_count())\n    for i in range(torch.cuda.device_count()):\n        props = torch.cuda.get_device_properties(i)\n        free, total = torch.cuda.mem_get_info(i)\n        print(f\'GPU {i}: {props.name}; free={free/1024**3:.2f} GiB / total={total/1024**3:.2f} GiB\')\n\n\n# Worker Config\n\nimport argparse\n\n@dataclass\nclass Config:\n    input_csv: str = \'/kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv\'\n    fallback_input_csv: str = \'court_considerations.csv\'\n    model_name: str = \'/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1\'\n\n    start: int = 0\n    limit: int = 50\n    sample_random: bool = False\n    random_seed: Optional[int] = 42\n    min_text_chars: int = 80\n    max_text_chars: int = 3200\n\n    gpu_mode: str = \'single\'\n    tensor_parallel_size: int = 1\n    gpu_memory_utilization: float = 0.88\n    max_model_len: int = 4096\n    max_num_seqs: int = 16\n    batch_size: int = 8\n    enforce_eager: bool = False\n    quantization: str = \'awq_marlin\'\n    disable_custom_all_reduce: bool = True\n    force_triton_attention: bool = True\n\n    use_structured_outputs: bool = False\n\n    max_new_tokens: int = 384\n    retry_max_new_tokens: int = 512\n    max_retries: int = 1\n    temperature: float = 0.0\n    top_p: float = 1.0\n    repetition_penalty: float = 1.02\n    enable_thinking: bool = False\n\n    output_dir: str = \'/kaggle/working\'\n    include_raw_output_on_success: bool = False\n\nparser = argparse.ArgumentParser()\nparser.add_argument(\'--gpu-id\', required=True)\nparser.add_argument(\'--start\', type=int, required=True)\nparser.add_argument(\'--limit\', type=int, required=True)\nparser.add_argument(\'--tag\', required=True)\nparser.add_argument(\'--model-name\', default=None)\nparser.add_argument(\'--input-csv\', default=None)\nparser.add_argument(\'--output-dir\', default=None)\nparser.add_argument(\'--batch-size\', type=int, default=None)\nparser.add_argument(\'--max-num-seqs\', type=int, default=None)\nparser.add_argument(\'--gpu-memory-utilization\', type=float, default=None)\nparser.add_argument(\'--max-model-len\', type=int, default=None)\nparser.add_argument(\'--max-new-tokens\', type=int, default=None)\nargs = parser.parse_args()\n\n# This must be set before vLLM initializes.\nos.environ[\'CUDA_VISIBLE_DEVICES\'] = str(args.gpu_id)\nos.environ.setdefault(\'TOKENIZERS_PARALLELISM\', \'false\')\nos.environ.setdefault(\'VLLM_WORKER_MULTIPROC_METHOD\', \'spawn\')\n\ncfg = Config()\ncfg.start = args.start\ncfg.limit = args.limit\ncfg.tensor_parallel_size = 1\nif args.model_name:\n    cfg.model_name = args.model_name\nif args.input_csv:\n    cfg.input_csv = args.input_csv\nif args.output_dir:\n    cfg.output_dir = args.output_dir\nif args.batch_size is not None:\n    cfg.batch_size = args.batch_size\nif args.max_num_seqs is not None:\n    cfg.max_num_seqs = args.max_num_seqs\nif args.gpu_memory_utilization is not None:\n    cfg.gpu_memory_utilization = args.gpu_memory_utilization\nif args.max_model_len is not None:\n    cfg.max_model_len = args.max_model_len\nif args.max_new_tokens is not None:\n    cfg.max_new_tokens = args.max_new_tokens\n\nif cfg.force_triton_attention:\n    os.environ.setdefault(\'VLLM_ATTENTION_BACKEND\', \'TRITON_ATTN\')\n\nmodel_path = Path(cfg.model_name)\nif not model_path.exists():\n    raise FileNotFoundError(f\'Model path does not exist: {model_path}. Attach the Kaggle model dataset.\')\n\nout_dir = Path(cfg.output_dir)\nout_dir.mkdir(parents=True, exist_ok=True)\n\nsuffix = args.tag\noutput_jsonl = out_dir / f\'court_llm_descriptors_{suffix}.jsonl\'\noutput_preview_csv = out_dir / f\'court_llm_descriptors_{suffix}_preview.csv\'\noutput_failures_jsonl = out_dir / f\'court_llm_descriptors_{suffix}_failures.jsonl\'\noutput_metrics_json = out_dir / f\'court_llm_descriptors_{suffix}_metrics.json\'\n\nprint(asdict(cfg), flush=True)\nprint(\'Visible CUDA devices:\', os.environ.get(\'CUDA_VISIBLE_DEVICES\'), flush=True)\nprint(\'Output JSONL:\', output_jsonl, flush=True)\n\n\n# Cell 3 — Load CSV and select rows\n\ndef resolve_input_path() -> Path:\n    candidates = [\n        Path(cfg.input_csv),\n        Path(cfg.fallback_input_csv),\n        Path(\'/kaggle/working\') / cfg.fallback_input_csv,\n    ]\n    for p in candidates:\n        if p.exists():\n            return p\n    if Path(\'/kaggle/input\').exists():\n        for pat in [\'**/court_considerations.csv\', \'**/court_consideration.csv\']:\n            found = sorted(Path(\'/kaggle/input\').glob(pat))\n            if found:\n                return found[0]\n    raise FileNotFoundError(\'Could not find court_considerations.csv\')\n\ninput_path = resolve_input_path()\nprint(\'Using input:\', input_path)\n\ndf = pd.read_csv(input_path)\nprint(\'Shape:\', df.shape)\nprint(\'Columns:\', list(df.columns))\n\ncitation_col = \'citation\' if \'citation\' in df.columns else df.columns[0]\ntext_col = \'text\' if \'text\' in df.columns else df.columns[1]\n\nvalid = df[df[citation_col].notna() & df[text_col].notna()].copy()\nvalid[text_col] = valid[text_col].astype(str)\nvalid[\'_text_len\'] = valid[text_col].str.strip().str.len()\nvalid = valid[valid[\'_text_len\'] >= cfg.min_text_chars].copy()\n\nif cfg.sample_random:\n    pool = valid.iloc[cfg.start:] if cfg.start else valid\n    work_df = pool.sample(n=min(cfg.limit, len(pool)), random_state=cfg.random_seed)\nelse:\n    end = None if not cfg.limit else cfg.start + cfg.limit\n    work_df = valid.iloc[cfg.start:end]\n\nwork_df = work_df.reset_index(drop=False).rename(columns={\'index\': \'_source_row\'})\nprint(\'Selected rows:\', len(work_df))\ndisplay(work_df[[\'_source_row\', citation_col, text_col, \'_text_len\']].head(20))\n\n\n# Cell 4 — Minimal LLM descriptor schema and prompt\n\n# The LLM intentionally does NOT produce anchors, outcomes, summaries, questions, or retrieval views.\n# Local CPU scripts will build those deterministically later.\n\nDESCRIPTOR_KEYS = [\n    \'legal_area\',\n    \'primary_domain\',\n    \'secondary_domain\',\n    \'legal_domain_path\',\n    \'topic\',\n    \'subtopic\',\n    \'micro_topic\',\n    \'concepts_en\',\n    \'terms_original\',\n    \'doctrinal_rule\',\n    \'legal_test\',\n    \'fact_pattern_tags\',\n    \'procedural_context\',\n    \'paragraph_role\',\n    \'authority_role\',\n    \'specificity_score\',\n]\n\nROLE_VALUES = {\n    \'holding\', \'reasoning\', \'facts\', \'procedural_history\', \'legal_standard\',\n    \'application\', \'citation\', \'costs\', \'notification\', \'disposition\', \'neutral\'\n}\n\nLLM_SCHEMA_HINT = {\n    \'legal_area\': \'broad English legal area, max 6 words\',\n    \'primary_domain\': \'stable domain, max 6 words\',\n    \'secondary_domain\': \'narrow domain, max 8 words\',\n    \'legal_domain_path\': [\'2-5 short taxonomy labels, broad to narrow\'],\n    \'topic\': \'specific topic, max 8 words\',\n    \'subtopic\': \'more specific subtopic, max 10 words\',\n    \'micro_topic\': \'most specific legal issue, max 14 words\',\n    \'concepts_en\': [\'3-6 precise English legal concepts\'],\n    \'terms_original\': [\'3-8 exact important source-language legal terms\'],\n    \'doctrinal_rule\': \'only if paragraph states a rule; otherwise empty; max 25 words\',\n    \'legal_test\': \'only if paragraph states/applies a test; otherwise empty; max 20 words\',\n    \'fact_pattern_tags\': [\'0-5 concrete factual/procedural tags\'],\n    \'procedural_context\': \'short procedural posture if clear\',\n    \'paragraph_role\': \'holding|reasoning|facts|procedural_history|legal_standard|application|citation|costs|notification|disposition|neutral\',\n    \'authority_role\': [\'0-3 legal value labels, e.g. legal_test, application_of_rule, background\'],\n    \'specificity_score\': \'number 0 to 1\',\n}\n\nSYSTEM_PROMPT = \'\'\'You are a Swiss legal descriptor extractor.\n\nReturn exactly one compact JSON object.\nDo not generate user questions.\nDo not generate summaries.\nDo not extract statute anchors.\nDo not extract case anchors.\nDo not infer final court outcome.\nDo not build retrieval views.\n\nOnly extract query-neutral semantic descriptors that cannot be reliably obtained by static regex parsing.\nUse English for classification fields.\nUse exact German/French/Italian terms for terms_original.\nIf the paragraph is factual/procedural/boilerplate, keep doctrinal_rule and legal_test empty.\nUse only information grounded in the citation text.\nJSON only.\'\'\'\n\nUSER_TEMPLATE = \'\'\'Citation: {citation}\n\nText:\n{text}\n\nRequired JSON shape:\n{schema}\n\nReturn JSON only.\'\'\'\n\ndef trim_text(text: str, max_chars: int) -> str:\n    text = re.sub(r\'\\s+\', \' \', str(text)).strip()\n    if len(text) <= max_chars:\n        return text\n    head = max_chars // 2\n    tail = max_chars - head\n    return text[:head].rstrip() + \' ... [TRUNCATED] ... \' + text[-tail:].lstrip()\n\ndef build_user_prompt(citation: str, text: str) -> str:\n    return USER_TEMPLATE.format(\n        citation=str(citation),\n        text=trim_text(text, cfg.max_text_chars),\n        schema=json.dumps(LLM_SCHEMA_HINT, ensure_ascii=False),\n    )\n\n\n# Cell 5 — JSON parsing and descriptor normalization\n\ndef extract_json_object(raw: str) -> dict[str, Any]:\n    if raw is None:\n        raise ValueError(\'empty model output\')\n    s = str(raw).strip()\n    s = re.sub(r\'^\\s*```(?:json)?\\s*\', \'\', s, flags=re.I)\n    s = re.sub(r\'\\s*```\\s*$\', \'\', s)\n    s = re.sub(r\'<think>.*?</think>\', \'\', s, flags=re.I | re.S).strip()\n\n    start = s.find(\'{\')\n    if start < 0:\n        raise ValueError(f\'no JSON object start found: {s[:300]}\')\n\n    depth = 0\n    in_str = False\n    esc = False\n    for i in range(start, len(s)):\n        ch = s[i]\n        if in_str:\n            if esc:\n                esc = False\n            elif ch == \'\\\\\':\n                esc = True\n            elif ch == \'"\':\n                in_str = False\n        else:\n            if ch == \'"\':\n                in_str = True\n            elif ch == \'{\':\n                depth += 1\n            elif ch == \'}\':\n                depth -= 1\n                if depth == 0:\n                    candidate = s[start:i+1]\n                    try:\n                        return json.loads(candidate)\n                    except Exception:\n                        candidate = re.sub(r\',\\s*([}\\]])\', r\'\\1\', candidate)\n                        try:\n                            return json.loads(candidate)\n                        except Exception:\n                            return ast.literal_eval(candidate)\n    raise ValueError(f\'no balanced JSON object found: {s[:700]}\')\n\n\ndef clean_str(x: Any, max_chars: int = 240) -> str:\n    s = re.sub(r\'\\s+\', \' \', str(x or \'\')).strip()\n    return s[:max_chars].rstrip()\n\n\ndef clean_list(x: Any, max_items: int, max_chars: int = 80) -> list[str]:\n    if x is None:\n        return []\n    if isinstance(x, str):\n        x = [x]\n    if not isinstance(x, (list, tuple, set)):\n        return []\n    out, seen = [], set()\n    for item in x:\n        s = clean_str(item, max_chars=max_chars)\n        if not s:\n            continue\n        key = s.casefold()\n        if key in seen:\n            continue\n        seen.add(key)\n        out.append(s)\n        if len(out) >= max_items:\n            break\n    return out\n\n\ndef normalize_descriptor(obj: dict[str, Any]) -> dict[str, Any]:\n    d = {}\n    d[\'legal_area\'] = clean_str(obj.get(\'legal_area\'), 80)\n    d[\'primary_domain\'] = clean_str(obj.get(\'primary_domain\'), 80)\n    d[\'secondary_domain\'] = clean_str(obj.get(\'secondary_domain\'), 100)\n    d[\'legal_domain_path\'] = clean_list(obj.get(\'legal_domain_path\'), 5, 60)\n    d[\'topic\'] = clean_str(obj.get(\'topic\'), 100)\n    d[\'subtopic\'] = clean_str(obj.get(\'subtopic\'), 120)\n    d[\'micro_topic\'] = clean_str(obj.get(\'micro_topic\'), 160)\n    d[\'concepts_en\'] = clean_list(obj.get(\'concepts_en\'), 6, 70)\n    d[\'terms_original\'] = clean_list(obj.get(\'terms_original\'), 8, 100)\n    d[\'doctrinal_rule\'] = clean_str(obj.get(\'doctrinal_rule\'), 260)\n    d[\'legal_test\'] = clean_str(obj.get(\'legal_test\'), 220)\n    d[\'fact_pattern_tags\'] = clean_list(obj.get(\'fact_pattern_tags\'), 5, 70)\n    d[\'procedural_context\'] = clean_str(obj.get(\'procedural_context\'), 120)\n\n    role = clean_str(obj.get(\'paragraph_role\'), 60).lower().replace(\' \', \'_\').replace(\'-\', \'_\')\n    d[\'paragraph_role\'] = role if role in ROLE_VALUES else \'neutral\'\n    d[\'authority_role\'] = clean_list(obj.get(\'authority_role\'), 3, 60)\n    try:\n        d[\'specificity_score\'] = max(0.0, min(1.0, float(obj.get(\'specificity_score\', 0))))\n    except Exception:\n        d[\'specificity_score\'] = 0.0\n\n    # Hard guarantee: forbidden/final fields never survive in raw descriptor.\n    forbidden = {\n        \'statute_anchors\', \'case_anchors\', \'normalized_anchors\', \'retrieval_views\',\n        \'outcome_signal\', \'query_phrases_en\', \'natural_language_queries\', \'legal_question\',\n        \'summary_en\', \'english_summary\', \'enrichment_quality\', \'anchor_quality_flags\',\n    }\n    for k in forbidden:\n        d.pop(k, None)\n    return d\n\n\ndef empty_descriptor(error: str = \'\') -> dict[str, Any]:\n    return {\n        \'legal_area\': \'\',\n        \'primary_domain\': \'\',\n        \'secondary_domain\': \'\',\n        \'legal_domain_path\': [],\n        \'topic\': \'\',\n        \'subtopic\': \'\',\n        \'micro_topic\': \'\',\n        \'concepts_en\': [],\n        \'terms_original\': [],\n        \'doctrinal_rule\': \'\',\n        \'legal_test\': \'\',\n        \'fact_pattern_tags\': [],\n        \'procedural_context\': \'\',\n        \'paragraph_role\': \'neutral\',\n        \'authority_role\': [],\n        \'specificity_score\': 0.0,\n        \'_descriptor_error\': error[:500],\n    }\n\n\n# Cell 6 — Load vLLM\n\nfrom transformers import AutoTokenizer\nfrom vllm import LLM, SamplingParams\n\nprint(\'Loading tokenizer/model:\', cfg.model_name)\ntokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)\n\nllm_kwargs = dict(\n    model=cfg.model_name,\n    trust_remote_code=True,\n    tensor_parallel_size=cfg.tensor_parallel_size,\n    gpu_memory_utilization=cfg.gpu_memory_utilization,\n    max_model_len=cfg.max_model_len,\n    max_num_seqs=cfg.max_num_seqs,\n    enforce_eager=cfg.enforce_eager,\n    quantization=cfg.quantization,\n    disable_custom_all_reduce=cfg.disable_custom_all_reduce,\n    disable_log_stats=True,\n)\n\n# vLLM attention config support differs by version.\nif cfg.force_triton_attention:\n    try:\n        from vllm.config import AttentionConfig\n        try:\n            llm_kwargs[\'attention_config\'] = AttentionConfig(backend=\'TRITON_ATTN\')\n        except Exception:\n            llm_kwargs[\'attention_config\'] = AttentionConfig(backend=\'triton_attn\')\n        print(\'[vLLM] using explicit TRITON attention_config\')\n    except Exception as exc:\n        print(\'[vLLM] AttentionConfig unavailable; env/default backend only:\', repr(exc))\n\nllm = LLM(**llm_kwargs)\nprint(\'vLLM loaded\')\n\n\n# Cell 7 — Generation helpers\n\ndef render_prompt(citation: str, text: str, repair: bool = False, bad_output: str = \'\', error: str = \'\') -> str:\n    user_prompt = build_user_prompt(citation, text)\n    if repair:\n        user_prompt = f\'\'\'The previous output was invalid JSON.\n\nParser error:\n{error}\n\nPrevious output:\n{bad_output[:1600]}\n\nRepair by returning exactly one complete compact JSON object using the same schema.\nDo not add questions, summaries, anchors, outcomes, or retrieval views.\n\n{user_prompt}\'\'\'\n\n    messages = [\n        {\'role\': \'system\', \'content\': SYSTEM_PROMPT},\n        {\'role\': \'user\', \'content\': user_prompt},\n    ]\n    try:\n        return tokenizer.apply_chat_template(\n            messages,\n            tokenize=False,\n            add_generation_prompt=True,\n            enable_thinking=cfg.enable_thinking,\n        )\n    except TypeError:\n        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n\n\ndef generate_raw(prompts: list[str], max_tokens: int) -> list[str]:\n    params = SamplingParams(\n        temperature=cfg.temperature,\n        top_p=cfg.top_p,\n        max_tokens=max_tokens,\n        repetition_penalty=cfg.repetition_penalty,\n    )\n    outputs = llm.generate(prompts, sampling_params=params, use_tqdm=False)\n    return [out.outputs[0].text if out.outputs else \'\' for out in outputs]\n\n\ndef generate_descriptor(citation: str, text: str, first_raw: str | None = None) -> tuple[dict[str, Any], dict[str, Any]]:\n    attempts = []\n    raw = first_raw\n    for attempt in range(cfg.max_retries + 1):\n        try:\n            if raw is None:\n                raw = generate_raw([render_prompt(citation, text)], cfg.max_new_tokens)[0]\n            obj = extract_json_object(raw)\n            desc = normalize_descriptor(obj)\n            return desc, {\n                \'status\': \'ok\' if attempt == 0 else \'ok_after_retry\',\n                \'attempt_count\': attempt + 1,\n                \'error\': None,\n                \'raw_output\': raw if cfg.include_raw_output_on_success else None,\n            }\n        except Exception as exc:\n            err = repr(exc)\n            attempts.append({\'attempt\': attempt + 1, \'error\': err, \'raw_output\': (raw or \'\')[:1600]})\n            if attempt >= cfg.max_retries:\n                return empty_descriptor(err), {\n                    \'status\': \'failed_descriptor_parse\',\n                    \'attempt_count\': attempt + 1,\n                    \'error\': err,\n                    \'attempts\': attempts,\n                    \'raw_output\': raw,\n                }\n            raw = generate_raw([render_prompt(citation, text, repair=True, bad_output=raw or \'\', error=err)], cfg.retry_max_new_tokens)[0]\n\n\n# Run descriptor extraction and write raw LLM descriptor JSONL\n\nrecords = []\nfailures = []\nt0 = time.time()\n\nfor batch_start in tqdm(range(0, len(work_df), cfg.batch_size), desc=f\'gpu{args.gpu_id}-llm-batches\'):\n    batch = work_df.iloc[batch_start:batch_start + cfg.batch_size]\n    row_objs = []\n    prompts = []\n    for _, row in batch.iterrows():\n        citation = str(row[citation_col])\n        text = str(row[text_col])\n        row_obj = {\n            \'_source_row\': int(row[\'_source_row\']),\n            \'citation\': citation,\n            \'text\': text,\n        }\n        row_objs.append(row_obj)\n        prompts.append(render_prompt(citation, text))\n\n    try:\n        raws = generate_raw(prompts, cfg.max_new_tokens)\n    except Exception as exc:\n        print(\'Batch generation failed; falling back to single-row generation:\', repr(exc), flush=True)\n        raws = [None] * len(row_objs)\n\n    for row_obj, raw in zip(row_objs, raws):\n        desc, gen = generate_descriptor(row_obj[\'citation\'], row_obj[\'text\'], first_raw=raw)\n        rec = {\n            \'_source_row\': row_obj[\'_source_row\'],\n            \'citation\': row_obj[\'citation\'],\n            \'text\': row_obj[\'text\'],\n            \'llm_enrichment\': desc,\n            \'llm_generation\': {\n                \'model\': cfg.model_name,\n                \'method\': \'minimal_descriptor_only\',\n                \'gpu_id\': str(args.gpu_id),\n                **gen,\n            },\n        }\n        records.append(rec)\n        if gen[\'status\'].startswith(\'failed\'):\n            failures.append(rec)\n\nelapsed = time.time() - t0\n\nwith output_jsonl.open(\'w\', encoding=\'utf-8\') as f:\n    for rec in records:\n        f.write(json.dumps(rec, ensure_ascii=False) + \'\\n\')\n\nif failures:\n    with output_failures_jsonl.open(\'w\', encoding=\'utf-8\') as f:\n        for rec in failures:\n            f.write(json.dumps(rec, ensure_ascii=False) + \'\\n\')\nelse:\n    if output_failures_jsonl.exists():\n        output_failures_jsonl.unlink()\n\npreview_rows = []\nfor rec in records:\n    e = rec[\'llm_enrichment\']\n    g = rec[\'llm_generation\']\n    preview_rows.append({\n        \'_source_row\': rec[\'_source_row\'],\n        \'citation\': rec[\'citation\'],\n        \'status\': g[\'status\'],\n        \'legal_area\': e.get(\'legal_area\'),\n        \'primary_domain\': e.get(\'primary_domain\'),\n        \'secondary_domain\': e.get(\'secondary_domain\'),\n        \'topic\': e.get(\'topic\'),\n        \'subtopic\': e.get(\'subtopic\'),\n        \'micro_topic\': e.get(\'micro_topic\'),\n        \'concepts_en\': \' | \'.join(e.get(\'concepts_en\', [])),\n        \'terms_original\': \' | \'.join(e.get(\'terms_original\', [])),\n        \'doctrinal_rule\': e.get(\'doctrinal_rule\'),\n        \'legal_test\': e.get(\'legal_test\'),\n        \'fact_pattern_tags\': \' | \'.join(e.get(\'fact_pattern_tags\', [])),\n        \'procedural_context\': e.get(\'procedural_context\'),\n        \'paragraph_role\': e.get(\'paragraph_role\'),\n        \'authority_role\': \' | \'.join(e.get(\'authority_role\', [])),\n        \'specificity_score\': e.get(\'specificity_score\'),\n    })\npreview_df = pd.DataFrame(preview_rows)\npreview_df.to_csv(output_preview_csv, index=False)\n\nmetrics = {\n    \'gpu_id\': str(args.gpu_id),\n    \'start\': cfg.start,\n    \'limit\': cfg.limit,\n    \'selected_rows\': len(work_df),\n    \'written_rows\': len(records),\n    \'failures\': len(failures),\n    \'elapsed_seconds\': elapsed,\n    \'rows_per_second\': len(records) / max(elapsed, 1e-9),\n    \'status_counts\': dict(Counter(rec[\'llm_generation\'][\'status\'] for rec in records)),\n    \'output_jsonl\': str(output_jsonl),\n    \'output_preview_csv\': str(output_preview_csv),\n    \'output_failures_jsonl\': str(output_failures_jsonl) if failures else None,\n}\noutput_metrics_json.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding=\'utf-8\')\nprint(json.dumps(metrics, indent=2), flush=True)\n', encoding='utf-8')
    print('Wrote worker script:', worker_script_path)

    first_count = (total_rows + 1) // 2
    second_count = total_rows - first_count

    shards = [
        {'gpu_id': '0', 'start': cfg.start, 'limit': first_count, 'tag': f'gpu0_{cfg.start:07d}_{cfg.start + first_count - 1:07d}'},
    ]
    if second_count > 0:
        shards.append({
            'gpu_id': '1',
            'start': cfg.start + first_count,
            'limit': second_count,
            'tag': f'gpu1_{cfg.start + first_count:07d}_{cfg.start + total_rows - 1:07d}',
        })

    procs = []
    log_paths = []
    for shard in shards:
        log_path = Path(cfg.output_dir) / f"vllm_worker_gpu{shard['gpu_id']}.log"
        cmd = [
            sys.executable,
            str(worker_script_path),
            '--gpu-id', shard['gpu_id'],
            '--start', str(shard['start']),
            '--limit', str(shard['limit']),
            '--tag', shard['tag'],
            '--model-name', cfg.model_name,
            '--input-csv', str(input_path),
            '--output-dir', cfg.output_dir,
            '--batch-size', str(cfg.batch_size),
            '--max-num-seqs', str(cfg.max_num_seqs),
            '--gpu-memory-utilization', str(cfg.gpu_memory_utilization),
            '--max-model-len', str(cfg.max_model_len),
            '--max-new-tokens', str(cfg.max_new_tokens),
        ]
        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = shard['gpu_id']
        env['PYTHONUNBUFFERED'] = '1'
        env.setdefault('TOKENIZERS_PARALLELISM', 'false')
        env.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')
        env.setdefault('VLLM_ATTENTION_BACKEND', 'TRITON_ATTN')

        print('Launching:', ' '.join(shlex.quote(x) for x in cmd))
        log_f = log_path.open('w', encoding='utf-8')
        p = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=env)
        procs.append((p, log_f, shard))
        log_paths.append(log_path)

    while True:
        statuses = [p.poll() for p, _, _ in procs]
        if all(s is not None for s in statuses):
            break
        alive = [shard['gpu_id'] for p, _, shard in procs if p.poll() is None]
        print('Workers still running on GPUs:', ', '.join(alive))
        time.sleep(30)

    for p, log_f, shard in procs:
        log_f.close()

    failures_proc = []
    for p, _, shard in procs:
        if p.returncode != 0:
            failures_proc.append((shard, p.returncode))

    if failures_proc:
        for shard, code in failures_proc:
            log_path = Path(cfg.output_dir) / f"vllm_worker_gpu{shard['gpu_id']}.log"
            print(f"\n--- Last 120 lines from {log_path} ---")
            if log_path.exists():
                lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
                print('\n'.join(lines[-120:]))
        raise RuntimeError(f"One or more dual_worker subprocesses failed: {failures_proc}")

    part_paths = [Path(cfg.output_dir) / f"court_llm_descriptors_{shard['tag']}.jsonl" for shard in shards]
    records = []
    for pth in part_paths:
        with pth.open('r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))

    records = sorted(records, key=lambda r: int(r.get('_source_row', -1)))

    with output_jsonl.open('w', encoding='utf-8') as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')

    failures = [rec for rec in records if str(rec.get('llm_generation', {}).get('status', '')).startswith('failed')]
    if failures:
        with output_failures_jsonl.open('w', encoding='utf-8') as f:
            for rec in failures:
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    elif output_failures_jsonl.exists():
        output_failures_jsonl.unlink()

    preview_rows = []
    for rec in records:
        e = rec['llm_enrichment']
        g = rec['llm_generation']
        preview_rows.append({
            '_source_row': rec['_source_row'],
            'citation': rec['citation'],
            'status': g['status'],
            'gpu_id': g.get('gpu_id'),
            'legal_area': e.get('legal_area'),
            'primary_domain': e.get('primary_domain'),
            'secondary_domain': e.get('secondary_domain'),
            'topic': e.get('topic'),
            'subtopic': e.get('subtopic'),
            'micro_topic': e.get('micro_topic'),
            'concepts_en': ' | '.join(e.get('concepts_en', [])),
            'terms_original': ' | '.join(e.get('terms_original', [])),
            'doctrinal_rule': e.get('doctrinal_rule'),
            'legal_test': e.get('legal_test'),
            'fact_pattern_tags': ' | '.join(e.get('fact_pattern_tags', [])),
            'procedural_context': e.get('procedural_context'),
            'paragraph_role': e.get('paragraph_role'),
            'authority_role': ' | '.join(e.get('authority_role', [])),
            'specificity_score': e.get('specificity_score'),
        })
    preview_df = pd.DataFrame(preview_rows)
    preview_df.to_csv(output_preview_csv, index=False)

    worker_metrics = []
    for shard in shards:
        pth = Path(cfg.output_dir) / f"court_llm_descriptors_{shard['tag']}_metrics.json"
        if pth.exists():
            worker_metrics.append(json.loads(pth.read_text(encoding='utf-8')))

    elapsed_max = max((m.get('elapsed_seconds', 0.0) for m in worker_metrics), default=0.0)
    metrics = {
        'mode': 'dual_worker',
        'start': cfg.start,
        'limit': cfg.limit,
        'selected_rows': len(work_df),
        'written_rows': len(records),
        'failures': len(failures),
        'elapsed_seconds_parallel_wall_estimate': elapsed_max,
        'rows_per_second_parallel_estimate': len(records) / max(elapsed_max, 1e-9),
        'status_counts': dict(Counter(rec['llm_generation']['status'] for rec in records)),
        'worker_metrics': worker_metrics,
        'part_paths': [str(p) for p in part_paths],
        'worker_logs': [str(p) for p in log_paths],
        'output_jsonl': str(output_jsonl),
        'output_preview_csv': str(output_preview_csv),
        'output_failures_jsonl': str(output_failures_jsonl) if failures else None,
    }
    output_metrics_json.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')

    DUAL_WORKER_DONE = True
    print(json.dumps(metrics, indent=2))
    display(preview_df)
else:
    DUAL_WORKER_DONE = False
    print("cfg.gpu_mode is not 'dual_worker'; continue with the normal single/tp2 cells.")


In [7]:
# Cell 6c — Load vLLM for normal single/tp2 mode

if cfg.gpu_mode == 'dual_worker':
    print("Skipping in-notebook vLLM load because dual_worker already ran subprocess workers.")
    tokenizer = None
    llm = None
else:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    print('Loading tokenizer/model:', cfg.model_name)
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)

    llm_kwargs = dict(
        model=cfg.model_name,
        trust_remote_code=True,
        tensor_parallel_size=cfg.tensor_parallel_size,
        gpu_memory_utilization=cfg.gpu_memory_utilization,
        max_model_len=cfg.max_model_len,
        max_num_seqs=cfg.max_num_seqs,
        enforce_eager=cfg.enforce_eager,
        quantization=cfg.quantization,
        disable_custom_all_reduce=cfg.disable_custom_all_reduce,
        disable_log_stats=True,
    )

    # vLLM attention config support differs by version.
    if cfg.force_triton_attention:
        try:
            from vllm.config import AttentionConfig
            try:
                llm_kwargs['attention_config'] = AttentionConfig(backend='TRITON_ATTN')
            except Exception:
                llm_kwargs['attention_config'] = AttentionConfig(backend='triton_attn')
            print('[vLLM] using explicit TRITON attention_config')
        except Exception as exc:
            print('[vLLM] AttentionConfig unavailable; env/default backend only:', repr(exc))

    llm = LLM(**llm_kwargs)
    print('vLLM loaded')


Loading tokenizer/model: /kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1
[vLLM] using explicit TRITON attention_config
INFO 05-03 07:32:10 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'gpu_memory_utilization': 0.78, 'max_num_seqs': 8, 'disable_log_stats': True, 'quantization': 'awq_marlin', 'enforce_eager': True, 'disable_custom_all_reduce': True, 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.TRITON_ATTN: 'vllm.v1.attention.backends.triton_attn.TritonAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, tq_max_kv_splits_for_cuda_graph=32, use_cudnn_prefill=False, use_trtllm_ragged_deepseek_prefill=False, use_trtllm_attention=None, disable_flashinfer_prefill=True, disable_flashinfer_q_quantization=False, use_prefill_query_quantization=False, use_fp4_indexer_cache=False), 'model': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1'}
WARNING 

[W503 07:32:48.562628145 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


(EngineCore pid=446) INFO 05-03 07:32:49 [gpu_model_runner.py:4777] Starting to load model /kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1...
(EngineCore pid=446) INFO 05-03 07:32:49 [awq_marlin.py:420] Using MarlinLinearKernel for AWQMarlinLinearMethod
(EngineCore pid=446) INFO 05-03 07:32:49 [cuda.py:308] Using AttentionBackendEnum.TRITON_ATTN backend.
(EngineCore pid=446) INFO 05-03 07:32:49 [weight_utils.py:904] Filesystem type for checkpoints: NFS. Checkpoint size: 5.68 GiB. Available RAM: 19.38 GiB.
(EngineCore pid=446) INFO 05-03 07:32:49 [weight_utils.py:874] Prefetching checkpoint files into page cache started (in background)


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=446) INFO 05-03 07:33:00 [weight_utils.py:851] Prefetching checkpoint files: 10% (1/2)


Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:53<00:53, 53.50s/it]


(EngineCore pid=446) INFO 05-03 07:33:43 [weight_utils.py:851] Prefetching checkpoint files: 20% (2/2)
(EngineCore pid=446) INFO 05-03 07:33:43 [weight_utils.py:869] Prefetching checkpoint files into page cache finished in 54.15s


Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:55<00:00, 22.92s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:55<00:00, 27.50s/it]
(EngineCore pid=446) 


(EngineCore pid=446) INFO 05-03 07:33:44 [default_loader.py:384] Loading weights took 55.10 seconds
(EngineCore pid=446) INFO 05-03 07:33:47 [gpu_model_runner.py:4879] Model loading took 5.71 GiB memory and 57.270029 seconds
(EngineCore pid=446) INFO 05-03 07:34:07 [gpu_worker.py:440] Available KV cache memory: 4.79 GiB
(EngineCore pid=446) INFO 05-03 07:34:07 [kv_cache_utils.py:1711] GPU KV cache size: 34,848 tokens
(EngineCore pid=446) INFO 05-03 07:34:07 [kv_cache_utils.py:1716] Maximum concurrency for 4,096 tokens per request: 8.51x
(EngineCore pid=446) INFO 05-03 07:34:07 [core.py:306] init engine (profile, create kv cache, warmup model) took 20.42 s
(EngineCore pid=446) INFO 05-03 07:34:08 [vllm.py:840] Asynchronous scheduling is enabled.
(EngineCore pid=446) WARNING 05-03 07:34:08 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=446) WARNING 05-03 07:34:08 [vllm.py:914] In

In [8]:
# Cell 7 — Generation helpers

def render_prompt(citation: str, text: str, repair: bool = False, bad_output: str = '', error: str = '') -> str:
    user_prompt = build_user_prompt(citation, text)
    if repair:
        user_prompt = f'''The previous output was invalid JSON.

Parser error:
{error}

Previous output:
{bad_output[:1600]}

Repair by returning exactly one complete compact JSON object using the same schema.
Do not add questions, summaries, anchors, outcomes, or retrieval views.

{user_prompt}'''

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=cfg.enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_raw(prompts: list[str], max_tokens: int) -> list[str]:
    params = SamplingParams(
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        max_tokens=max_tokens,
        repetition_penalty=cfg.repetition_penalty,
    )
    outputs = llm.generate(prompts, sampling_params=params, use_tqdm=False)
    return [out.outputs[0].text if out.outputs else '' for out in outputs]


def generate_descriptor(citation: str, text: str, first_raw: str | None = None) -> tuple[dict[str, Any], dict[str, Any]]:
    attempts = []
    raw = first_raw
    for attempt in range(cfg.max_retries + 1):
        try:
            if raw is None:
                raw = generate_raw([render_prompt(citation, text)], cfg.max_new_tokens)[0]
            obj = extract_json_object(raw)
            desc = normalize_descriptor(obj)
            return desc, {
                'status': 'ok' if attempt == 0 else 'ok_after_retry',
                'attempt_count': attempt + 1,
                'error': None,
                'raw_output': raw if cfg.include_raw_output_on_success else None,
            }
        except Exception as exc:
            err = repr(exc)
            attempts.append({'attempt': attempt + 1, 'error': err, 'raw_output': (raw or '')[:1600]})
            if attempt >= cfg.max_retries:
                return empty_descriptor(err), {
                    'status': 'failed_descriptor_parse',
                    'attempt_count': attempt + 1,
                    'error': err,
                    'attempts': attempts,
                    'raw_output': raw,
                }
            raw = generate_raw([render_prompt(citation, text, repair=True, bad_output=raw or '', error=err)], cfg.retry_max_new_tokens)[0]


In [ ]:
# Cell 8 — Run descriptor extraction and write raw LLM descriptor JSONL

if cfg.gpu_mode == 'dual_worker':
    print("Skipping in-notebook extraction because dual_worker has already written and merged:")
    print(" ", output_jsonl)
    if 'records' not in globals():
        records = []
        if output_jsonl.exists():
            with output_jsonl.open('r', encoding='utf-8') as f:
                for line in f:
                    if line.strip():
                        records.append(json.loads(line))
        failures = [rec for rec in records if str(rec.get('llm_generation', {}).get('status', '')).startswith('failed')]
    if 'preview_df' in globals():
        display(preview_df)
else:
    records = []
    failures = []
    t0 = time.time()

    for start in tqdm(range(0, len(work_df), cfg.batch_size), desc='llm-descriptor batches'):
        batch = work_df.iloc[start:start + cfg.batch_size]
        row_objs = []
        prompts = []
        for _, row in batch.iterrows():
            citation = str(row[citation_col])
            text = str(row[text_col])
            row_obj = {
                '_source_row': int(row['_source_row']),
                'citation': citation,
                'text': text,
            }
            row_objs.append(row_obj)
            prompts.append(render_prompt(citation, text))

        try:
            raws = generate_raw(prompts, cfg.max_new_tokens)
        except Exception as exc:
            print('Batch generation failed; falling back to single-row generation:', repr(exc))
            raws = [None] * len(row_objs)

        for row_obj, raw in zip(row_objs, raws):
            desc, gen = generate_descriptor(row_obj['citation'], row_obj['text'], first_raw=raw)
            rec = {
                '_source_row': row_obj['_source_row'],
                'citation': row_obj['citation'],
                'text': row_obj['text'],
                'llm_enrichment': desc,
                'llm_generation': {
                    'model': cfg.model_name,
                    'method': 'minimal_descriptor_only',
                    **gen,
                },
            }
            records.append(rec)
            if gen['status'].startswith('failed'):
                failures.append(rec)

    elapsed = time.time() - t0

    with output_jsonl.open('w', encoding='utf-8') as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')

    if failures:
        with output_failures_jsonl.open('w', encoding='utf-8') as f:
            for rec in failures:
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    else:
        if output_failures_jsonl.exists():
            output_failures_jsonl.unlink()

    preview_rows = []
    for rec in records:
        e = rec['llm_enrichment']
        g = rec['llm_generation']
        preview_rows.append({
            '_source_row': rec['_source_row'],
            'citation': rec['citation'],
            'status': g['status'],
            'legal_area': e.get('legal_area'),
            'primary_domain': e.get('primary_domain'),
            'secondary_domain': e.get('secondary_domain'),
            'topic': e.get('topic'),
            'subtopic': e.get('subtopic'),
            'micro_topic': e.get('micro_topic'),
            'concepts_en': ' | '.join(e.get('concepts_en', [])),
            'terms_original': ' | '.join(e.get('terms_original', [])),
            'doctrinal_rule': e.get('doctrinal_rule'),
            'legal_test': e.get('legal_test'),
            'fact_pattern_tags': ' | '.join(e.get('fact_pattern_tags', [])),
            'procedural_context': e.get('procedural_context'),
            'paragraph_role': e.get('paragraph_role'),
            'authority_role': ' | '.join(e.get('authority_role', [])),
            'specificity_score': e.get('specificity_score'),
        })
    preview_df = pd.DataFrame(preview_rows)
    preview_df.to_csv(output_preview_csv, index=False)

    metrics = {
        'start': cfg.start,
        'limit': cfg.limit,
        'selected_rows': len(work_df),
        'written_rows': len(records),
        'failures': len(failures),
        'elapsed_seconds': elapsed,
        'rows_per_second': len(records) / max(elapsed, 1e-9),
        'status_counts': dict(Counter(rec['llm_generation']['status'] for rec in records)),
        'output_jsonl': str(output_jsonl),
        'output_preview_csv': str(output_preview_csv),
        'output_failures_jsonl': str(output_failures_jsonl) if failures else None,
    }
    output_metrics_json.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')

    print(json.dumps(metrics, indent=2))
    display(preview_df)


llm-descriptor batches:   0%|          | 0/13 [00:00<?, ?it/s]

In [ ]:
# Cell 9 — QC: verify this is raw descriptor output only

FORBIDDEN = {
    'statute_anchors', 'case_anchors', 'normalized_anchors', 'retrieval_views',
    'outcome_signal', 'query_phrases_en', 'natural_language_queries', 'legal_question',
    'summary_en', 'english_summary', 'enrichment_quality', 'anchor_quality_flags',
}

def find_forbidden(obj: Any, path: str = '') -> list[str]:
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f'{path}.{k}' if path else k
            if k in FORBIDDEN:
                hits.append(p)
            hits.extend(find_forbidden(v, p))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hits.extend(find_forbidden(v, f'{path}[{i}]'))
    return hits

qc = []
for rec in records:
    e = rec['llm_enrichment']
    qc.append({
        'citation': rec['citation'],
        'status': rec['llm_generation']['status'],
        'forbidden_fields': find_forbidden(rec),
        'concept_count': len(e.get('concepts_en', [])),
        'terms_original_count': len(e.get('terms_original', [])),
        'has_topic': bool(e.get('topic') or e.get('subtopic') or e.get('micro_topic')),
        'specificity_score': e.get('specificity_score'),
    })
qc_df = pd.DataFrame(qc)
display(qc_df)
print('Forbidden field rows:', int(qc_df['forbidden_fields'].apply(bool).sum()))
print('Failed rows:', int(qc_df['status'].str.startswith('failed').sum()))


## Next local step

Download `court_llm_descriptors_*.jsonl` and run the local finalizer, which should call:

```python
normalize_enriched_court_row(
    citation=citation,
    text=text,
    llm_enrichment=raw["llm_enrichment"],
    deterministic_metadata=metadata,
)
```

That local step creates the final production JSONL with:

```text
rag_enrichment
normalized_anchors
anchor_quality_flags
retrieval_views
enrichment_quality
```
